# DIS 2026: train GPT-2 from scratch on Shakespeare text with JAX + TPU

This notebook is the student-facing workshop path. We will:

1. verify that JAX sees a Cloud TPU;
2. prepare text from the Tiny Shakespeare corpus;
3. tokenize with the GPT-2 tokenizer;
4. build a GPT-2-style model from random initialization;
5. train it with JAX/Flax/Optax on TPU;
6. measure compile time and steady-state throughput;
7. sample Shakespeare-like text;
8. clean up the TPU.

Source adapted from Wei Wei / windmaple's Google Developers tutorial:

- https://developers.googleblog.com/train-gpt2-model-with-jax-on-tpu/
- https://github.com/windmaple/LLM_from_scratch.JAX/tree/main/02.GPT2-pretraining

Important: this is **pretraining from scratch**, not fine-tuning pretrained GPT-2 weights.


## 0. Install dependencies

On a TPU VM, run this cell once. The TPU-specific JAX install line follows current Cloud TPU practice. If the environment already has JAX/Flax installed, this is safe to skip.


In [ ]:
# If running interactively in Jupyter, run this once on the TPU VM if imports fail.
# From repo root use requirements-tpu.txt; from notebooks/ use ../requirements-tpu.txt.
# %pip install -U pip
# %pip install -r requirements-tpu.txt
# # or, if the current working directory is notebooks/:
# %pip install -r ../requirements-tpu.txt


## 1. TPU sanity check

First confirm we are on the right machine and JAX sees TPU devices. The first compiled operation is slow because XLA compiles for the TPU; the second run shows steady-state behavior.


In [ ]:
import os, time, json, math, random
from pathlib import Path

import jax
import jax.numpy as jnp

print("JAX version:", jax.__version__)
print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())

@jax.jit
def matmul_smoke(x):
    return x @ x

x = jnp.ones((2048, 2048), dtype=jnp.float32)
t0 = time.perf_counter()
y = matmul_smoke(x).block_until_ready()
t1 = time.perf_counter()
y = matmul_smoke(x).block_until_ready()
t2 = time.perf_counter()
print("result shape:", y.shape)
print(f"first run incl compile: {t1-t0:.3f}s")
print(f"second run compiled:   {t2-t1:.3f}s")


## 2. The JAX training-step mental model

PyTorch often feels like:

```text
model object -> loss.backward() -> optimizer.step()
```

JAX makes the data flow more explicit:

```text
params/state + batch -> loss
value_and_grad(loss) -> gradients
optax.update -> parameter update
jit(train_step) -> compiled TPU step
```

The GPT-2 training loop below keeps those pieces visible.


## 3. Prepare Shakespeare tokens

This uses the repo helper `scripts/prepare_shakespeare_tokens.py`. The helper downloads the Tiny Shakespeare text corpus, chunks it into short training records, and tokenizes it with the GPT-2 tokenizer.

For class, start with a subset so compile/training/debug cycles are fast. For a longer run, increase `MAX_RECORDS` or remove it.


In [ ]:
# In the repository root, this notebook expects ../scripts/prepare_shakespeare_tokens.py.
# If you copied only the notebook, copy the scripts/ folder too.
import subprocess, sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts" / "prepare_shakespeare_tokens.py").exists() and (REPO_ROOT.parent / "scripts" / "prepare_shakespeare_tokens.py").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data" / "shakespeare_tokens_smoke"
MAX_RECORDS = 2000  # increase for longer runs

prep_script = REPO_ROOT / "scripts" / "prepare_shakespeare_tokens.py"
if not prep_script.exists():
    raise FileNotFoundError(f"Missing token-prep script: {prep_script}")

inspect_cmd = [
    sys.executable,
    str(prep_script),
    "--inspect",
    "--max-records",
    "5",
]
print("Inspecting a few Shakespeare examples before tokenization:")
print(" ".join(inspect_cmd))
subprocess.run(inspect_cmd, check=True)

cmd = [
    sys.executable,
    str(prep_script),
    "--out-dir",
    str(DATA_DIR),
    "--max-records",
    str(MAX_RECORDS),
]
print("Tokenization command:")
print(" ".join(cmd))
if not (DATA_DIR / "train.bin").exists() or not (DATA_DIR / "val.bin").exists():
    subprocess.run(cmd, check=True)
else:
    print(f"Token files already exist in {DATA_DIR}")


In [ ]:
# Inspect metadata after running the data-prep command.
meta_path = DATA_DIR / "metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print(json.dumps(meta, indent=2))
else:
    print("metadata.json not found yet. Run the data prep command above.")


## 4. Imports and configuration

The default `tiny` config is deliberately smaller than full GPT-2 so everyone can get a complete run. The architecture is still GPT-2-style: token embeddings, positional embeddings, causal multi-head self-attention, MLP blocks, weight tying, autoregressive cross-entropy.


In [ ]:
import numpy as np
import tiktoken
import optax
import flax.nnx as nnx
# Sharding is an extension topic; the default notebook path lets JAX place arrays.

TOKENIZER = tiktoken.get_encoding("gpt2")
VOCAB_SIZE = TOKENIZER.n_vocab

MODEL_SIZE = "tiny"  # "tiny", "small", or "gpt2ish"

CONFIGS = {
    "tiny": dict(num_transformer_blocks=4, seqlen=256, embed_dim=256, num_heads=4, batch_size=16),
    "small": dict(num_transformer_blocks=8, seqlen=512, embed_dim=512, num_heads=8, batch_size=16),
    # Full-ish GPT-2 124M shape; use only when quota/time/budget allow.
    "gpt2ish": dict(num_transformer_blocks=12, seqlen=1024, embed_dim=768, num_heads=12, batch_size=8),
}

cfg = CONFIGS[MODEL_SIZE]
num_transformer_blocks = cfg["num_transformer_blocks"]
seqlen = cfg["seqlen"]
embed_dim = cfg["embed_dim"]
num_heads = cfg["num_heads"]
batch_size = cfg["batch_size"]
feed_forward_dim = 4 * embed_dim

dropout_rate = 0.1
init_learning_rate = 5e-4
weight_decay = 1e-1
top_k = 20
dtype = jnp.bfloat16
param_dtype = jnp.float32
max_steps = 100  # workshop-safe default; increase after the short run works
log_every = 10

print(cfg)


## 5. Load token streams and define batches

This follows the upstream/nanoGPT style: randomly choose offsets in a flat token stream, then predict the next token at each position.


In [ ]:
train_bin = DATA_DIR / "train.bin"
val_bin = DATA_DIR / "val.bin"
if not train_bin.exists() or not val_bin.exists():
    raise FileNotFoundError("Run the Shakespeare token preparation cell first.")

train_data = np.memmap(train_bin, dtype=np.uint16, mode="r")
val_data = np.memmap(val_bin, dtype=np.uint16, mode="r")
print("train tokens:", len(train_data), "val tokens:", len(val_data))

rng_np = np.random.default_rng(0)

def get_batch(split="train"):
    data = train_data if split == "train" else val_data
    if len(data) <= seqlen + 1:
        raise ValueError(f"Not enough tokens in {split}: {len(data)} <= {seqlen+1}")
    ix = rng_np.integers(0, len(data) - seqlen - 1, size=(batch_size,))
    x = np.stack([data[i:i+seqlen].astype(np.int64) for i in ix])
    y = np.stack([data[i+1:i+1+seqlen].astype(np.int64) for i in ix])
    return x, y

xb, yb = get_batch("train")
print(xb.shape, yb.shape, TOKENIZER.decode(xb[0, :80].tolist()))


## 6. GPT-2-style model definition

Adapted from the upstream `GPT2_pretrain.ipynb`. We use Flax NNX modules and Optax. The key JAX point: the model state is transformed by a compiled training step.


In [ ]:
def causal_attention_mask(seq_len):
    return jnp.tril(jnp.ones((seq_len, seq_len), dtype=bool))

class TransformerBlock(nnx.Module):
    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, dropout_rate: float, rngs: nnx.Rngs):
        self.layer_norm1 = nnx.LayerNorm(
            epsilon=1e-6, num_features=embed_dim, dtype=dtype, param_dtype=param_dtype, rngs=rngs
        )
        self.mha = nnx.MultiHeadAttention(
            num_heads=num_heads, in_features=embed_dim, dtype=dtype, param_dtype=param_dtype, rngs=rngs
        )
        self.dropout1 = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.layer_norm2 = nnx.LayerNorm(
            epsilon=1e-6, num_features=embed_dim, dtype=dtype, param_dtype=param_dtype, rngs=rngs
        )
        self.linear1 = nnx.Linear(embed_dim, ff_dim, dtype=dtype, param_dtype=param_dtype, rngs=rngs)
        self.linear2 = nnx.Linear(ff_dim, embed_dim, dtype=dtype, param_dtype=param_dtype, rngs=rngs)
        self.dropout2 = nnx.Dropout(rate=dropout_rate, rngs=rngs)

    def __call__(self, inputs, training: bool = False):
        seq_len = inputs.shape[1]
        attention_output = self.mha(
            inputs_q=self.layer_norm1(inputs),
            mask=causal_attention_mask(seq_len),
            decode=False,
        )
        x = inputs + self.dropout1(attention_output, deterministic=not training)
        mlp_output = self.linear1(self.layer_norm2(x))
        mlp_output = nnx.gelu(mlp_output)
        mlp_output = self.linear2(mlp_output)
        mlp_output = self.dropout2(mlp_output, deterministic=not training)
        return x + mlp_output

class TokenAndPositionEmbedding(nnx.Module):
    def __init__(self, seqlen: int, vocab_size: int, embed_dim: int, rngs: nnx.Rngs):
        self.token_emb = nnx.Embed(vocab_size, embed_dim, dtype=dtype, param_dtype=param_dtype, rngs=rngs)
        self.pos_emb = nnx.Embed(seqlen, embed_dim, dtype=dtype, param_dtype=param_dtype, rngs=rngs)

    def __call__(self, x):
        positions = jnp.arange(0, x.shape[1])[None, :]
        return self.token_emb, self.token_emb(x) + self.pos_emb(positions)

class GPT2(nnx.Module):
    def __init__(self, seqlen, vocab_size, embed_dim, num_heads, rate, feed_forward_dim, num_transformer_blocks, rngs: nnx.Rngs):
        self.embedding_layer = TokenAndPositionEmbedding(seqlen, vocab_size, embed_dim, rngs=rngs)
        self.dropout = nnx.Dropout(rate=rate, rngs=rngs)
        self.transformer_blocks = nnx.List([
            TransformerBlock(embed_dim, num_heads, feed_forward_dim, rate, rngs=rngs)
            for _ in range(num_transformer_blocks)
        ])
        self.layer_norm = nnx.LayerNorm(
            epsilon=1e-6, num_features=embed_dim, dtype=dtype, param_dtype=param_dtype, rngs=rngs
        )

    def __call__(self, inputs, training: bool = False):
        token_embedding, x = self.embedding_layer(inputs)
        x = self.dropout(x, deterministic=not training)
        for block in self.transformer_blocks:
            x = block(x, training=training)
        x = self.layer_norm(x)
        return token_embedding.attend(x)  # weight tying

    @nnx.jit
    def generate_step(self, padded_tokens, sample_index, key):
        logits = self(padded_tokens, training=False)
        logits, indices = jax.lax.top_k(logits[0][sample_index], k=top_k)
        probs = jax.nn.softmax(logits)
        return jax.random.choice(key, indices, p=probs)

    def generate_text(self, prompt, max_new_tokens=80, seed=0):
        start_tokens = TOKENIZER.encode(prompt)[:seqlen]
        generated = []
        key = jax.random.PRNGKey(seed)
        for i in range(max_new_tokens):
            key, subkey = jax.random.split(key)
            sample_index = len(start_tokens) + len(generated) - 1
            if sample_index >= seqlen - 1:
                break
            padded = jnp.array(start_tokens + generated + [0] * (seqlen - len(start_tokens) - len(generated)))[None, :]
            next_token = int(self.generate_step(padded, sample_index, subkey))
            if next_token == TOKENIZER.eot_token:
                break
            generated.append(next_token)
        return TOKENIZER.decode(start_tokens + generated)

def create_model(rngs):
    return GPT2(seqlen, VOCAB_SIZE, embed_dim, num_heads, dropout_rate, feed_forward_dim, num_transformer_blocks, rngs=rngs)


## 7. Loss, optimizer, and compiled training step

This is the core JAX lesson. The loss is an ordinary function of model + batch; `nnx.value_and_grad` transforms it to return gradients; Optax defines the update rule; `nnx.Optimizer` owns the mutable optimizer state; and `optimizer.update(model, grads)` mutates the model parameters/optimizer state in the NNX object graph. `nnx.jit` compiles the whole step for TPU once shapes and dtypes are stable.


In [ ]:
# Default path: let JAX place arrays on the available device(s).
# Explicit meshes/sharding are an extension after the basic TPU run works.

def loss_fn(model, batch, training: bool = True):
    tokens, targets = batch
    logits = model(tokens, training=training)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=targets).mean()
    return loss, logits

@nnx.jit
def train_step(model: nnx.Module, optimizer: nnx.Optimizer, batch):
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model, batch, True)
    optimizer.update(model, grads)
    return loss

schedule = optax.cosine_decay_schedule(init_value=init_learning_rate, decay_steps=max_steps)
optimizer_def = optax.adamw(learning_rate=schedule, weight_decay=weight_decay)
model = create_model(rngs=nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optimizer_def, wrt=nnx.Param)

param_count = sum(x.size for x in jax.tree.leaves(nnx.state(model)) if hasattr(x, 'size'))
print(f"parameter count: {param_count:,}")
print("initial sample:")
print(model.generate_text("User: can you fix this bug?\nAssistant:", max_new_tokens=80, seed=0))


## 8. Train and measure throughput

The first step includes compilation. Subsequent steps are the steady-state TPU path.


In [ ]:
metrics = []
compile_seconds = None
steady_start = None
steady_tokens = 0

for step in range(max_steps):
    x_np, y_np = get_batch("train")
    batch = jax.device_put((x_np, y_np))
    t0 = time.perf_counter()
    loss = train_step(model, optimizer, batch)
    loss.block_until_ready()
    dt = time.perf_counter() - t0
    if step == 0:
        compile_seconds = dt
        steady_start = time.perf_counter()
    else:
        steady_tokens += batch_size * seqlen
    if step % log_every == 0:
        elapsed = 0.0 if steady_start is None else time.perf_counter() - steady_start
        tok_s = steady_tokens / elapsed if elapsed > 0 else 0.0
        print(f"step={step:05d} loss={float(loss):.4f} step_time={dt:.3f}s steady_tokens_per_s={tok_s:,.0f}")
        metrics.append(dict(step=step, loss=float(loss), step_time=dt, steady_tokens_per_s=tok_s))

print(f"compile step seconds: {compile_seconds:.3f}")
print("final sample:")
print(model.generate_text("User: can you fix this bug?\nAssistant:", max_new_tokens=120, seed=1))


## 9. Validation sample and optional checkpoint

For a short workshop run, sampling is more important than checkpointing. For longer runs, save with Orbax after the class demo.


In [ ]:
# Optional validation loss. Very small smoke subsets may not have enough validation tokens.
if len(val_data) > seqlen + 1:
    xv, yv = get_batch("val")
    val_loss, _ = loss_fn(model, jax.device_put((xv, yv)), False)
    print("val_loss", float(val_loss))
else:
    print(f"validation skipped: only {len(val_data)} validation tokens for seqlen={seqlen}")

# Optional checkpoint skeleton:
# import orbax.checkpoint as orbax
# ckpt_dir = Path.home() / "dis_2026_gpt2_shakespeare_checkpoint"
# ckpt_dir.mkdir(exist_ok=True)
# checkpointer = orbax.PyTreeCheckpointer()
# checkpointer.save(str(ckpt_dir), nnx.to_pure_dict(nnx.state(model)))


## 10. Cleanup

A READY TPU costs money even if idle. Before you leave, delete it.

From your local machine:

```bash
gcloud compute tpus tpu-vm list --project="$PROJECT_ID" --zone="$ZONE"
gcloud compute tpus tpu-vm delete "$TPU_NAME" --project="$PROJECT_ID" --zone="$ZONE" --quiet
gcloud compute tpus tpu-vm list --project="$PROJECT_ID" --zone="$ZONE"
```

If in doubt, use the panic cleanup in `TPU_AGENT.md`.
